In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import string


import os
# import tensorflow as tf

from keras.applications.xception import Xception,preprocess_input   #transfer models


import pickle

from tqdm import tqdm_notebook as tqdm   #this show progress bar for training loops

from PIL import Image




ModuleNotFoundError: No module named 'keras'

In [ ]:
tqdm().pandas()  #applying progress bar on pandas

In [ ]:
# Loading Text file

def load_doc(file_name):
    with open(file_name,"r") as f:
        text = f.read() #reading entire file at once
        f.close()  #closhing file read mode 

    return text
    

In [ ]:
def all_img_caption(file_name):
    file = load_doc(file_name)
    captions = file.split("\n")
    description = {}

    for line in captions:
        img, caption = line.split("\t")

        img_id = img.split("#")[0]   

        if img_id not in description:
            description[img_id] = [caption]
        else:
            description[img_id].append(caption)

    return description   

In [ ]:
# Cleaning text
def cleaning_text(caption):
    table = str.maketrans("", "", string.punctuation)

    for img, caps in caption.items():
        for i in range(len(caps)):

            img_caption = caps[i]

            # remove hyphen
            img_caption = img_caption.replace("-"," ")

            # split into words
            words = img_caption.split()

            cleaned_words = []

            for word in words:
                word = word.lower()         # lowercase
                word = word.translate(table)    # remove punctuation

                if len(word) > 1:               # remove small words like a i specially single character word
                    cleaned_words.append(word)

            # join back
            caps[i] = " ".join(cleaned_words)  

    return caption

In [ ]:
# Tokenization

def text_vocab(description):
    vocab = set()

    for key in description.keys():
        for sentence in description[key]:
            words = sentence.split()

            for word in words:
                vocab.update(word)  #ipdate convert word hello into  --> h,e,l,o 

    return vocab
                
                    
    

In [ ]:
# # Tesing method

# test_data = {
#     "img1": ["this image belong to dog"]
# }

# res = text_vocab(test_data)
# print(res)

In [ ]:
# Saving clean description in same or new file 

def save_description(description,file_name):
    lines = list()
    for key,desc_list in description.items():
        for desc in desc_list:
            lines.append(key +"\t" + desc)  #mean store image  description format

    data = "\n".join(lines)  #split all into different line

    with open(file=file_name,mode="w") as f:
        f.write(data)
        f.close()
        

In [ ]:

print(os.listdir("/home/darshan/Deep Learning/Dataset/Flickr8k_Dataset"))

In [ ]:
file_name ="/home/darshan/Deep Learning/Dataset/Flickr8k_Dataset/Flickr8k_text/Flickr8k.token.txt"
description = all_img_caption(file_name)


In [ ]:
description

In [ ]:
clean_description = cleaning_text(description)
clean_description

In [ ]:
vocab = text_vocab(clean_description)
print(vocab)

In [ ]:
len(vocab)

In [ ]:
save_description(clean_description,"cleaned_captions.txt")


In [ ]:
# Dataset FOlder PAth
path = "Deep Learning/Dataset/Flickr8k_Dataset"

In [ ]:
# USing model 
model = Xception(weights="imagenet")


In [ ]:
model.summary()

In [ ]:
# Since we are generating caption we need to know the information of whole frame so we will use avg plloing in plcae of max pooling for smooth process
model = Xception(weights='imagenet', include_top=False,pooling="avg")  #since cnn model have classifier and feature so while using  Pretrained Cnn we disable the classifier and our own similarly we disable the Classiffer in this model also

In [ ]:
# model total trainable parameter after removing Classifier part
model.summary()

In [ ]:
def extracting_features(file_path_img, model, batch_size=32):
    features = {}
    valid_image = ['.jpg', '.png', '.jpeg']

    batch_images = []
    batch_names = []

    for img in tqdm(os.listdir(file_path_img)):
        extension = os.path.splitext(img)[1].lower()  #split hello.jpg to hello , jpg

        if extension not in valid_image:
            continue

        file_name = os.path.join(file_path_img, img)  #picking each img from file

        # Load image
        image = Image.open(file_name).convert('RGB')

        # Resize for Xception
        image = image.resize((299, 299))

        # Convert to array
        image = np.array(image)

        # Add to batch
        batch_images.append(image)
        batch_names.append(img)

        # When batch is full → process
        if len(batch_images) == batch_size:
            batch_images_np = np.array(batch_images)

            # Preprocess
            batch_images_np = preprocess_input(batch_images_np)  #normalize

            # Predict
            batch_features = model.predict(batch_images_np, verbose=0)

            # Store
            for i, name in enumerate(batch_names):  #mean img.jpg = [predicted matrix]
                features[name] = batch_features[i]

            # Reset batch
            batch_images = []
            batch_names = []

    # Process remaining images which can not form batch due to less than 32 image
    if batch_images:
        batch_images_np = np.array(batch_images)
        batch_images_np = preprocess_input(batch_images_np)

        batch_features = model.predict(batch_images_np, verbose=0)

        for i, name in enumerate(batch_names):
            features[name] = batch_features[i]

    return features

In [ ]:

print(tf.config.list_physical_devices('GPU'))




In [ ]:
features = extracting_features("/home/darshan/Deep Learning/Dataset/Flickr8k_Dataset/Flicker8k_Dataset",model)


with open("features.p", "wb") as f:
    pickle.dump(features, f)
    
                               

In [ ]:
with open("features.p","rb") as f:    #cause we store in binary form so we reading file in binary mode
    features = pickle.load(f)

In [ ]:
features

In [ ]:
def loading_photo(file_name,dataset_images):
    file = load_doc(file_name)
    images = file.split("\n")[:-1]   #when image end at end  there is the blank space so removing it 
    photos_present = []

    for photo in images:
        full_path = os.path.join(dataset_images, photo)
    
        if os.path.exists(full_path):
            photos_present.append(photo)

    return photos_present
    

In [ ]:
def load_clean_description(file_name,photo):
    file = load_doc(file_name)
    description ={}

    for line in file.split("\n"):
        words = line.split()
        if len(words)<1:
            continue

        image = words[0]
        image_caption = words[1:]

        if image in photo:
            if image not in photo:
                description[image] = []
    
            # our RNN model need Start and end token to indicate the start and end of words
            desc = "<start>"+ join(image_caption) + "<End>"
    
            description[image].append(desc)

    return description




            
        
        
        
        
    

In [ ]:
def load_features(photo):
    with open("features.p", "rb") as f:
        features = pickle.load(f)

    result = {}

    for k in photo:
        result[k] = features[k]


    return result

In [ ]:
file_name = "/home/darshan/Deep Learning/Dataset/Flickr8k_Dataset/Flickr8k_text/Flickr_8k.trainImages.txt"

dataset_path = "/home/darshan/Deep Learning/Dataset/Flickr8k_Dataset/Flickr8k_Dataset"

train_img = loading_photo(file_name, dataset_path)

train_description = load_clean_description("cleaned_captions.txt",train_img)

train_features = load_features(train_img)


In [ ]:
def dict_to_list(description):
    all_desc = []
    for key in description.keys():
        for content in description[key]:
            all_desc.append(content)

    return all_desc
    
        

In [ ]:
def create_token(description):
    desc_list = dict_to_list(description)
    tokenizer = Tokenizer()
    tokenizer.fit_on_texts(desc_list)
    return tokenizer
    
    

In [ ]:
tokenizer = create_token(train_description)

with open("tokenizer.p", "wb") as f:
    pickle.dump(tokenizer, f)

In [ ]:
vocab_size = len(tokenizer.word_index)+1
print(vocab_size)


In [ ]:
def max_lenght_description(description):
    desc_list = dict_to_list(description)
    for i in desc_list:      
        return max(len(i.split()))

In [ ]:

max_lenght = max_lenght_description(train_description)
print(max_lenght)

In [ ]:
def create_sequences(tokenizer, max_length, desc_list, feature):
    X1, X2, y = [], [], []

    for desc in desc_list:
        seq = tokenizer.texts_to_sequences([desc])[0]

        for i in range(1, len(seq)):
            in_seq = seq[:i]
            out_seq = seq[i]

            in_seq = pad_sequences([in_seq], maxlen=max_length)[0]

            out_seq = to_categorical(
                [out_seq],
                num_classes=len(tokenizer.word_index) + 1
            )[0]

            X1.append(feature)
            X2.append(in_seq)
            y.append(out_seq)

    return np.array(X1), np.array(X2), np.array(y)

In [ ]:
def data_generator_simple(descriptions, features, tokenizer, max_length):
    input_image = []
    input_sequence = []
    output_word = []

    for key, description_list in descriptions.items():
        feature = features[key][0]

        X1, X2, y = create_sequences(
            tokenizer, max_length, description_list, feature
        )

        input_image.extend(X1)
        input_sequence.extend(X2)
        output_word.extend(y)

    X = {
        'input_1': np.array(input_image),
        'input_2': np.array(input_sequence)
    }
    y = np.array(output_word)

    return X, y

In [ ]:
X, y = data_generator_simple(
    train_description,
    train_features,
    tokenizer,
    max_lenght
)

dataset = tf.data.Dataset.from_tensor_slices((X, y))
dataset = dataset.batch(32)

In [ ]:
dataset

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# -----------------------------------
# Caption Model
# -----------------------------------
class CaptionModel(nn.Module):
    def __init__(self, vocab_size, max_length):
        super(CaptionModel, self).__init__()

        # Image feature branch
        self.dropout_img = nn.Dropout(0.5)
        self.fc_img = nn.Linear(2048, 256)

        # Text branch
        self.embedding = nn.Embedding(vocab_size, 256, padding_idx=0)
        self.dropout_txt = nn.Dropout(0.5)
        self.lstm = nn.LSTM(256, 256, batch_first=True)

        # Final layers
        self.fc1 = nn.Linear(256, 256)
        self.fc2 = nn.Linear(256, vocab_size)

    def forward(self, image, sequence):
        # -----------------------
        # Image branch
        # -----------------------
        img = self.dropout_img(image)
        img = F.relu(self.fc_img(img))   # (batch, 256)

        # -----------------------
        # Text branch
        # -----------------------
        seq = self.embedding(sequence)   # (batch, max_len, 256)
        seq = self.dropout_txt(seq)

        _, (hidden, _) = self.lstm(seq)
        seq = hidden[-1]  # (batch, 256)

        # -----------------------
        # Merge
        # -----------------------
        merged = img + seq

        x = F.relu(self.fc1(merged))
        output = self.fc2(x)  # (batch, vocab_size)

        return output